In [1]:
# -*- coding: utf-8 -*-
"""
둘 중 하나의 물리 기준을 고정해 비례 보정 후, 픽셀 해상도를 재계산하여 리사이징
- LOCK_MODE = "width": 100cm ↔ 3755px 고정 → 목표 해상도 3755×2629
- LOCK_MODE = "height": 70cm  ↔ 2649px 고정 → 목표 해상도 3784×2649
"""

import os
from math import isfinite
from PIL import Image

# ====== 사용자 설정 ======
INPUT_DIR  = r"C:\Users\_idal\PycharmProjects\Cloth_AI\data\1022_samples\crop"
OUTPUT_DIR = r"C:\Users\_idal\PycharmProjects\Cloth_AI\data\1022_samples\resize_RE"
OVERWRITE  = False  # True면 원본 위 덮어쓰기

# 물리 기준(문제에서 제시)
REF_W_CM, REF_H_CM = 100.0, 70.0
REF_W_PX, REF_H_PX = 3755, 2649

# 어떤 축을 고정할지 선택: "width" 또는 "height"
LOCK_MODE = "width"   # "height" 로 바꾸면 세로 고정 모드

# ====== 보정된 TARGET_SIZE 계산 ======
if LOCK_MODE.lower() == "width":
    # 가로 100cm ↔ 3755px을 정답으로 고정 → 같은 cm/px로 세로 70cm 환산
    cm_per_px_x = REF_W_CM / REF_W_PX           # 100/3755 cm/px
    height_px = round(REF_H_CM / cm_per_px_x)   # 70 / (100/3755) = 2628.5 → 2629
    TARGET_SIZE = (REF_W_PX, int(height_px))    # (3755, 2629)
elif LOCK_MODE.lower() == "height":
    # 세로 70cm ↔ 2649px을 정답으로 고정 → 같은 cm/px로 가로 100cm 환산
    cm_per_px_y = REF_H_CM / REF_H_PX           # 70/2649 cm/px
    width_px = round(REF_W_CM / cm_per_px_y)    # 100 / (70/2649) = 3784.29 → 3784
    TARGET_SIZE = (int(width_px), REF_H_PX)     # (3784, 2649)
else:
    raise ValueError("LOCK_MODE는 'width' 또는 'height' 중 하나여야 합니다.")

print(f"LOCK_MODE = {LOCK_MODE}")
print(f"계산된 TARGET_SIZE = {TARGET_SIZE[0]} x {TARGET_SIZE[1]} px")

# ====== 실행 ======
os.makedirs(OUTPUT_DIR, exist_ok=True)
valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_ext)]
print(f"총 {len(files)}개의 이미지가 감지되었습니다.")

for i, filename in enumerate(files, 1):
    input_path = os.path.join(INPUT_DIR, filename)
    output_path = input_path if OVERWRITE else os.path.join(OUTPUT_DIR, filename)

    try:
        with Image.open(input_path) as img:
            resized = img.resize(TARGET_SIZE, Image.LANCZOS)
            resized.save(output_path)
            print(f"[{i}/{len(files)}] {filename} → {TARGET_SIZE} 저장 완료")
    except Exception as e:
        print(f"[오류] {filename}: {e}")

print("✅ 비례 보정 후 리사이징 완료!")


LOCK_MODE = width
계산된 TARGET_SIZE = 3755 x 2628 px
총 5개의 이미지가 감지되었습니다.
[1/5] all-over pattern_rotate.jpg → (3755, 2628) 저장 완료
[2/5] check_1_rotate.jpg → (3755, 2628) 저장 완료
[3/5] print_1_rotate.jpg → (3755, 2628) 저장 완료
[4/5] stripe_1_rotate.jpg → (3755, 2628) 저장 완료
[5/5] stripe_2_rotate.jpg → (3755, 2628) 저장 완료
✅ 비례 보정 후 리사이징 완료!
